# Model comparison: caught cases, missed cases and false alerts

This notebook separates **model ranking** from **classification at an evaluation rule**.
It adds true positives (TP), false negatives (FN), false positives (FP), true negatives (TN),
precision, recall/sensitivity, specificity, negative predictive value and F1.

The two experiment profiles have different inputs and checklists. Compare algorithms
**within one profile and cohort**, keeping the held-out records and flag capacity fixed.
- `historical`: the broader completed-pregnancy research recipes in the original exports.
- `antenatal`: the corrected ongoing-pregnancy feature definitions used by the clinician API.

These tables reproduce the frozen export reports, not the live submitted metric rows.
Some live submissions differ from these refits. Do not attach these confusion matrices
to a different hosted run. The hosted store currently lacks experiment, flag-count and
confusion-matrix metadata. See `docs/model-comparison.md` for the read-only hosted check.

These tables reproduce the frozen export reports. Selected releases were calibrated;
other candidates were not. Calibration status is shown. This is not a new uniformly
calibrated training experiment and does not claim to reproduce every notebook verbatim.

This notebook does not publish metrics, replace fitted releases, change serving models,
or choose a winner using the test set. Team nominations are displayed for comparison only.

## Before running
Install the project ML dependencies in a compatible Python environment and provide the
licensed processed CSVs in `data/processed/`. The current antenatal artifacts can be installed
from the committed package with `python -m ml.src.package_release --install
backend/models/antenatal-release.tar.gz`. Historical comparison additionally needs the
corresponding local historical exports and their reports. No dataset or credentials are
embedded in this notebook. Each non-selected candidate is refitted using its recorded
training-CV hyperparameters; there is no new parameter search.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'ml/src/dataset.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ml.src.compare_classification import compare

PROFILE = 'antenatal'  # Change to 'historical' for the original export experiment.
RECOMPUTE = False
TEAM_NOMINATIONS = {
    'lbw': 'gradient_boosting',
    'perinatal': 'random_forest',
    'perinatal_anc': 'random_forest',
}


In [ ]:
destination = ROOT / 'ml/models/comparisons' / f'{PROFILE}.json'
if RECOMPUTE or not destination.exists():
    result = compare(PROFILE)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(result, indent=2) + '\n')
else:
    result = json.loads(destination.read_text())

assert result['profile'] == PROFILE
release_root = ROOT / 'ml/models' / ('antenatal' if PROFILE == 'antenatal' else '')
for group in result['cohorts']:
    release = release_root / group['cohort'] / 'selected.joblib'
    report = release.with_suffix('.json')
    for path, key in [(release, 'model_sha256'), (report, 'source_report_sha256')]:
        if hashlib.sha256(path.read_bytes()).hexdigest() != group.get(key):
            raise ValueError('Source release/report changed. Set RECOMPUTE=True and rerun.')
print('Profile:', result['profile'])
print(result['decision_rule'])
print(result['policy'])
print(result['calibration_note'])


## Read the comparison

The primary model-selection criterion in these existing releases was five-fold grouped
**training CV average precision** (reported as PR-AUC). `release_algorithm` identifies that
frozen selection. The following test results measure its subsequent performance.

PR-AUC/AP summarizes precision and recall across score cutoffs; it does not specify a
clinical action cutoff. ROC-AUC measures ranking across the two outcome groups. Precision,
recall and the confusion matrix depend on a binary flagging rule.

Here each model flags exactly as many test records as that profile's checklist.
This is a **research batch top-k comparison**. It is not a fixed probability cutoff for
assessing one new patient. Scores tied at the capacity limit use stable row order;
that tie rule can affect which individual records are flagged.


In [ ]:
rows = []
for group in result['cohorts']:
    for model in group['models']:
        rows.append({
            'cohort': group['cohort'],
            'algorithm': model['algorithm'],
            'team_nomination': model['algorithm'] == TEAM_NOMINATIONS[group['cohort']],
            'release_algorithm': model['is_release_algorithm'],
            'calibrated': model['calibrated'],
            'CV_AP': model['training_cv_average_precision'],
            'test_AP': model['pr_auc'], 'test_ROC_AUC': model['roc_auc'],
            'Brier_score': model['brier_score'],
            **{key: model[key] for key in ['tp', 'fn', 'fp', 'tn', 'flagged',
                                          'recall', 'precision', 'specificity',
                                          'negative_predictive_value', 'f1']},
        })
comparison = pd.DataFrame(rows)
display(comparison.round(4))


## Confusion matrices, including the checklist

Rows are the **recorded actual outcome**. Columns are the **model/checklist flag**.

| | Not flagged | Flagged |
|---|---|---|
| Actual outcome absent | TN | FP |
| Actual outcome present | FN | TP |

- Recall/sensitivity = TP / (TP + FN): how many affected cases were found?
- Precision/positive predictive value = TP / (TP + FP): how many flags were affected cases?
- Specificity = TN / (TN + FP): how many unaffected cases were not flagged?

A new patient's true outcome is unknown at assessment time. TP/FP/TN/FN can only be assigned
when the actual outcome is available. Undefined rates are null, not invented perfect scores.


In [ ]:
for group in result['cohorts']:
    print('\n', group['cohort'], ':', group['rows'], 'test records,', group['events'], 'events')
    for name, counts in [('checklist', group['checklist']),
                         *[(m['algorithm'], m) for m in group['models']]]:
        print(name)
        display(pd.DataFrame(
            [[counts['tn'], counts['fp']], [counts['fn'], counts['tp']]],
            index=['Actual outcome absent', 'Actual outcome present'],
            columns=['Not flagged', 'Flagged'],
        ))


## Which metric should guide the choice?

Choose the objective **before examining test rankings**. Both LBW and perinatal outcomes
are uncommon in these data, so report the full precision-recall tradeoff.

- For screening at a fixed review capacity, compare recall/TP and FN at the same number of
  flags, while reporting precision and FP to show the workload.
- For overall ranking of affected cases, use PR-AUC/AP and keep ROC-AUC as supporting evidence.
- For displayed individual probability estimates, also evaluate calibration and Brier score.
- For an operational positive/negative decision, agree the objective and permissible review
  workload with the clinical team, tune the rule on validation/training folds, freeze it,
  and then evaluate it once on untouched test data.

A high recall alone does not establish a useful model: flagging everybody gives 100% recall
and also flags every unaffected case. A small score difference alone does not establish a
reliable improvement; uncertainty and stability across folds matter.

The team nominations above are hypotheses to check against this evidence. They are not
forced results. No clinical threshold or deployment winner is changed by this notebook.

Sources: [scikit-learn metric definitions](https://scikit-learn.org/stable/modules/model_evaluation.html)
and [decision-threshold guidance](https://scikit-learn.org/stable/modules/classification_threshold.html).
